# 01 — Data Collection
Scrapes stock specs from ultimatespecs.com and modification figures from goapr.com.
Outputs:
- `data/raw/cars_raw.csv`
- `data/raw/modifications_raw.csv`

In [1]:
import sys
sys.path.insert(0, '..')  # makes src/ importable from notebooks/

import pandas as pd
from src.scraper import (
    scrape_ultimatespecs_car,
    scrape_ultimatespecs_batch,
    scrape_goapr_product,
    scrape_goapr_batch,
    save_csv,
    append_csv,
)

## 1. Stock specs — ultimatespecs.com

In [5]:
# ADD URLS HERE
CAR_URLS = [
    # VW Golf GTI Mk7 220hp
    #"https://www.ultimatespecs.com/car-specs/Volkswagen/65173/",
    # VW Golf GTI Mk7 230hp Performance
    #"https://www.ultimatespecs.com/car-specs/Volkswagen/65179/",
    # Honda Civic Type R FK8
    #"https://www.ultimatespecs.com/car-specs/Honda/M8969/Civic-10-5-doors",
    # Ford Focus ST Mk3 — find URL on ultimatespecs.com and add here
    "https://www.parkers.co.uk/ford/focus/st-2012/20t-st-3-estate-(0115-)-5d/specs/"
    # BMW 135i E82 — find URL on ultimatespecs.com and add here
]

cars_df = scrape_ultimatespecs_batch(CAR_URLS)
cars_df

INFO  GET https://www.parkers.co.uk/ford/focus/st-2012/20t-st-3-estate-(0115-)-5d/specs/
WARNING  No spec rows found — the page structure may have changed.
WARNING  Save the page HTML and inspect it to update the selectors.


,source_url,raw_title,hp_stock,torque_nm_stock,weight_kg,zero_to_100_stock,top_speed_stock,drag_coefficient,drivetrain,engine_cc,_raw
0,https://www.parkers.co.uk/ford/focus/st-2012/2...,Ford Focus ST2.0T ST-3 Estate (01/15-) 5d Spec...,None,None,None,None,None,None,None,None,{}


In [ ]:
# inspect the _raw dict for the first car to see what keys the scraper found.
# in the get(...) aliases inside scrape_ultimatespecs_car().
if not cars_df.empty:
    print(cars_df.iloc[0]['_raw'])

In [ ]:
# drop the debug column before saving
cars_clean = cars_df.drop(columns=['_raw'], errors='ignore')
save_csv(cars_clean, '../data/raw/cars_raw.csv')
cars_clean

## 2. Modifications — goapr.com

In [ ]:
# APR product pages — IS20 and IS38 ECU tunes for the GTI
APR_URLS = [
    "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html",
    "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is38.html",
    # Add Revo / Mountune / Unitronic URLs here
]

mods_df = scrape_goapr_batch(APR_URLS)
mods_df

In [ ]:
save_csv(mods_df, '../data/raw/modifications_raw.csv')
print(f"Saved {len(mods_df)} modification rows")

## 3. Builds — manual entry

Forum and YouTube builds need manual collection for now (login walls, JavaScript rendering).
Add rows here and they'll be appended to `data/raw/builds_raw.csv`.

In [ ]:
# Paste rows from forums/YouTube here.
# mod_description = raw text from the post, don't clean it — that's for notebook 02.
# confidence rubric:
#   0.90 — professional tuner dyno on their site
#   0.75 — forum user with dyno sheet image
#   0.60 — forum user claims dyno, no proof
#   0.35 — user estimate, no dyno
#   0.40 — physics-calculated synthetic

MANUAL_BUILDS = [
    {
        "car_id": "vw_golf_gti_mk7_220",
        "mod_description": "APR Stage 1 ECU tune only, no hardware mods",
        "result_hp": 307,
        "result_torque_nm": None,
        "result_0_100": None,
        "result_top_speed": None,
        "measurement_type": "dyno",
        "hp_type": "whp",
        "source_type": "tuner_site",
        "confidence": 0.90,
        "source_url": "https://www.goapr.com/products/ecu_upgrade_2-0t_gen3_mqb_is20.html",
    },
    # Add more rows here
]

builds_df = append_csv(MANUAL_BUILDS, '../data/raw/builds_raw.csv')
builds_df

## 4. Quick sanity check

In [2]:
for name, path in [("cars", '../data/raw/cars_raw.csv'), ("mods", '../data/raw/modifications_raw.csv'), ("builds", '../data/raw/builds_raw.csv')]:
    try:
        df = pd.read_csv(path)
        print(f"{name}: {len(df)} rows, {df.shape[1]} cols")
    except FileNotFoundError:
        print(f"{name}: not created yet")

cars: not created yet
mods: not created yet
builds: not created yet
